In [0]:
dbutils.widgets.text("env", "dev")
env     = dbutils.widgets.get("env")
BR      = f"{env}_bronze"
SRC     = "samples.tpch"
regions = ["africa", "america", "asia", "europe", "middle_east"]
def load(target, select_sql, source):
    spark.sql(f"""
        INSERT OVERWRITE {target}
        SELECT q.*, current_timestamp(), '{source}' FROM ({select_sql}) q
    """)
    print(f"✔ {target}: {spark.table(target).count():,} rows")


In [0]:
for t in ["region", "nation", "part", "supplier", "partsupp"]:
    load(f"{BR}.common.{t}", f"SELECT * FROM {SRC}.{t}", f"{SRC}.{t}")


In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW customer_region AS
SELECT c.c_custkey,
       lower(replace(trim(r.r_name), ' ', '_')) AS region_schema   -- e.g. 'middle_east'
FROM {SRC}.customer c
JOIN {SRC}.nation n ON c.c_nationkey = n.n_nationkey
JOIN {SRC}.region r ON n.n_regionkey = r.r_regionkey
""")
display(spark.sql("SELECT region_schema, COUNT(*) AS customers FROM customer_region GROUP BY 1 ORDER BY 1"))


In [0]:
for r in regions:
    load(f"{BR}.{r}.customer", f"""
        SELECT c.* FROM {SRC}.customer c
        JOIN customer_region cr ON c.c_custkey = cr.c_custkey
        WHERE cr.region_schema = '{r}'""", f"{SRC}.customer")
    load(f"{BR}.{r}.orders", f"""
        SELECT o.* FROM {SRC}.orders o
        JOIN customer_region cr ON o.o_custkey = cr.c_custkey
        WHERE cr.region_schema = '{r}'""", f"{SRC}.orders")
    load(f"{BR}.{r}.lineitem", f"""
        SELECT l.* FROM {SRC}.lineitem l
        JOIN {SRC}.orders o        ON l.l_orderkey = o.o_orderkey
        JOIN customer_region cr    ON o.o_custkey  = cr.c_custkey
        WHERE cr.region_schema = '{r}'""", f"{SRC}.lineitem")


In [0]:
for t in ["customer", "orders", "lineitem"]:
    src = spark.table(f"{SRC}.{t}").count()
    tgt = sum(spark.table(f"{BR}.{r}.{t}").count() for r in regions)
    print(f"{t:10} source={src:,}  bronze={tgt:,}  {'✅ OK' if src == tgt else '❌ MISMATCH'}")
